In [ ]:
%%sql -r dataframe_1
-- =====================================================
-- PHASE 1: SKYPOINTS INCUBYTE ENVIRONMENT
-- NOTEBOOK: NB_01_MEMBER_INGESTION
-- =====================================================

-- 1. Create warehouse
CREATE WAREHOUSE IF NOT EXISTS SKYPOINTS_WH
    WAREHOUSE_SIZE = 'XSMALL'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    INITIALLY_SUSPENDED = TRUE
    COMMENT = 'Warehouse for SkyPoints Incubyte Data Engineering Assessment';

-- 2. Create database
CREATE DATABASE IF NOT EXISTS SKYPOINTS_INCU
    COMMENT = 'SkyPoints Incubyte Data Engineering Project';

-- 3. Select warehouse and database
USE WAREHOUSE SKYPOINTS_WH;
USE DATABASE SKYPOINTS_INCU;

-- 4. Create schemas
CREATE SCHEMA IF NOT EXISTS RAW;
CREATE SCHEMA IF NOT EXISTS STG;
CREATE SCHEMA IF NOT EXISTS CORE;
CREATE SCHEMA IF NOT EXISTS MART;
CREATE SCHEMA IF NOT EXISTS OPS;

-- 5. Create member internal stage
CREATE STAGE IF NOT EXISTS RAW.MEMBER_STAGE
    DIRECTORY = (ENABLE = TRUE)
    COMMENT = 'Landing stage for daily member profile files';

-- 6. Create redemption internal stage
CREATE STAGE IF NOT EXISTS RAW.REDEMPTION_STAGE
    DIRECTORY = (ENABLE = TRUE)
    COMMENT = 'Landing stage for daily redemption JSON files';

-- 7. Verify database objects
SHOW SCHEMAS IN DATABASE SKYPOINTS_INCU;

SHOW STAGES IN SCHEMA SKYPOINTS_INCU.RAW;

SHOW WAREHOUSES LIKE 'SKYPOINTS_WH';

In [ ]:
%%sql -r dataframe_2
-- =====================================================
-- PHASE 2A: RAW MEMBER LANDING TABLE
-- NOTEBOOK: NB_01_MEMBER_INGESTION
-- =====================================================

USE WAREHOUSE SKYPOINTS_WH;
USE DATABASE SKYPOINTS_INCU;
USE SCHEMA RAW;

CREATE TABLE IF NOT EXISTS RAW_MEMBER_PROFILES (
    RAW_RECORD             VARIANT,
    SOURCE_FILE_NAME       VARCHAR(500) NOT NULL,
    FILE_TIMESTAMP         TIMESTAMP_NTZ,
    INGESTION_TIMESTAMP    TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    BATCH_ID               VARCHAR(100) NOT NULL,
    SOURCE_ROW_NUMBER      NUMBER(38,0)
);

-- Verify
DESCRIBE TABLE RAW_MEMBER_PROFILES;

SHOW TABLES IN SCHEMA RAW;

In [ ]:
%%sql -r dataframe_3
-- =====================================================
-- PHASE 2B: MEMBER CSV FILE FORMAT
-- NOTEBOOK: NB_01_MEMBER_INGESTION
-- =====================================================

USE WAREHOUSE SKYPOINTS_WH;
USE DATABASE SKYPOINTS_INCU;
USE SCHEMA RAW;

CREATE FILE FORMAT IF NOT EXISTS RAW.MEMBER_CSV_FORMAT
    TYPE = CSV
    FIELD_DELIMITER = ','
    SKIP_HEADER = 1
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    EMPTY_FIELD_AS_NULL = TRUE
    NULL_IF = ('NULL', 'null')
    TRIM_SPACE = TRUE
    ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE
    ENCODING = 'UTF8';

-- Verify file format
DESCRIBE FILE FORMAT RAW.MEMBER_CSV_FORMAT;

In [ ]:
%%sql -r dataframe_5
-- =====================================================
-- PHASE 3B: MEMBER RAW INGESTION
-- NOTEBOOK: NB_01_MEMBER_INGESTION
-- =====================================================

USE WAREHOUSE SKYPOINTS_WH;
USE DATABASE SKYPOINTS_INCU;

COPY INTO RAW.RAW_MEMBER_PROFILES
(
    RAW_RECORD,
    SOURCE_FILE_NAME,
    FILE_TIMESTAMP,
    INGESTION_TIMESTAMP,
    BATCH_ID,
    SOURCE_ROW_NUMBER
)
FROM (
    SELECT
        OBJECT_CONSTRUCT_KEEP_NULL(
            'MEMBER_NAME', $1,
            'MEMBER_ID', $2,
            'ENROLLMENT_DATE', $3,
            'LAST_FLIGHT_DATE', $4,
            'TIER_CODE', $5,
            'AGENT_NAME', $6,POST_CODE
            'STATE', $7,
            'COUNTRY', $8,
            'POST_CODE', $9,
            'DOB', $10,
            'ACTIVE_MEMBER', $11
        ),
        METADATA$FILENAME,
        TO_TIMESTAMP_NTZ(
            REGEXP_SUBSTR(
                METADATA$FILENAME,
                '[0-9]{8}_[0-9]{6}'
            ),
            'YYYYMMDD_HH24MISS'
        ),
        CURRENT_TIMESTAMP()::TIMESTAMP_NTZ,
        UUID_STRING(),
        METADATA$FILE_ROW_NUMBER
    FROM @RAW.MEMBER_STAGE
        (FILE_FORMAT => 'RAW.MEMBER_CSV_FORMAT')
)
PATTERN = '.*MEMBER_PROFILE_[0-9]{8}_[0-9]{6}[.]csv'
ON_ERROR = 'ABORT_STATEMENT';

In [ ]:
%%sql -r dataframe_7
-- =====================================================
-- PHASE 3C: STANDARDIZE MEMBER BATCH ID
-- =====================================================

UPDATE RAW.RAW_MEMBER_PROFILES
SET BATCH_ID = MD5(
    SOURCE_FILE_NAME || '|' ||
    TO_VARCHAR(FILE_TIMESTAMP, 'YYYYMMDD_HH24MISS')
)
WHERE SOURCE_FILE_NAME = 'MEMBER_PROFILE_20261125_193100.csv';

-- Verify batch consistency
SELECT
    SOURCE_FILE_NAME,
    BATCH_ID,
    COUNT(*) AS RECORD_COUNT,
    COUNT(DISTINCT BATCH_ID) AS DISTINCT_BATCH_IDS
FROM RAW.RAW_MEMBER_PROFILES
WHERE SOURCE_FILE_NAME = 'MEMBER_PROFILE_20261125_193100.csv'
GROUP BY SOURCE_FILE_NAME, BATCH_ID;